In [47]:
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import LuceneIndexReader
import json
from tqdm import tqdm
import re

In [17]:
searcher = LuceneSearcher("hotpotqa_index")
index_reader = LuceneIndexReader("hotpotqa_index")

In [18]:
index_reader.dump_documents_BM25("data/hotpotqa_bm25_vectors.jsonl")

100%|██████████| 415838/415838 [05:18<00:00, 1306.84it/s]


In [19]:
with open("data/hotpotqa_bm25_vectors.jsonl") as f, open("data/hotpotqa_bm25_vectors_dict.json", "w") as f_out:
    output_dict = {}
    lines = f.readlines()
    for i, line in tqdm(enumerate(lines), desc="Processing BM25 vectors", total=len(lines)):
        bm25_data = json.loads(line)
        output_dict[bm25_data['id']] = bm25_data['vector']
    json.dump(output_dict, f_out, indent=4)

Processing BM25 vectors: 100%|██████████| 415838/415838 [00:07<00:00, 52418.60it/s]


In [60]:
id_to_term = {}
term_to_id = {}
count = 0
idx = 0
for t in index_reader.terms():
    if t.df == t.cf:
        count += 1
        continue
    if not re.fullmatch(r"[A-Za-z]+", t.term):
        count += 1
        continue
    if t.cf <= 5:
        count += 1
        continue
    id_to_term[idx] = t.term
    term_to_id[t.term] = idx
    idx += 1

In [61]:
count, len(id_to_term)

(336943, 60301)

In [62]:
with open("data/hotpotqa_vocab_id_to_term.json", "w") as f:
    json.dump(id_to_term, f, indent=4)

with open("data/hotpotqa_vocab_term_to_id.json", "w") as f:
    json.dump(term_to_id, f, indent=4)